# Attention Profiling on GPU

## 1. Verify/Install Correct NSight Version

### Add CUDA binaries to PATH if needed

In [ ]:
import os
os.environ['PATH'] = os.pathsep.join([os.environ['PATH'], '/usr/local/cuda/bin/'])

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
!ncu --version

**I used:**

For RTX 2060 GPU:
* CUDA V13.2.51
* NSight 2026.1.0.0

For T4 GPU:
* CUDA V12.8.93
* NSight 2025.1.1

## 2. Testing Script

Make sure it works before running it all at once with NSight

### Setup

In [ ]:
!pip install numpy pandas

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

In [ ]:
# following FlashAttention-3 paper
def generate_matrix(shape, seed=None) -> np.ndarray:
    if seed is not None:
        np.random.seed(seed)
    # Base matrix from N(0, 1)
    base = np.random.normal(loc=0.0, scale=1.0, size=shape)
    # Bernoulli mask (0.001 probability of being 1)
    mask = np.random.binomial(n=1, p=0.001, size=shape)
    # Noise from N(0, 100)
    noise = np.random.normal(loc=0.0, scale=10.0, size=shape)
    # Final matrix: base + noise * mask
    return base + noise * mask

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
device

In [ ]:
def scaled_dot_product_attention(Q_np: np.ndarray, K_np: np.ndarray, V_np: np.ndarray, causal: bool) -> np.ndarray:
    # ensure matching dimensions of 4D tensors
    assert (len(Q_np.shape), len(K_np.shape), len(V_np.shape)) == (4, 4, 4)
    b, h, seq_q, d = Q_np.shape
    bk, hk, seq_k, dk = K_np.shape
    bv, hv, seq_v, dv = V_np.shape
    assert b == 1 and b == bk and b == bv
    assert h == 1 and h == hk and h == hv
    assert d == dk, "Q and K head dim must be equal"
    assert d == dv, f"Q ({d}) and V ({dv}) head dim must be equal"
    assert seq_k == seq_v, "K and V must have equal seq len"

    # use CUDA on GPU
    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    Q_torch = torch.from_numpy(Q_np).to(device)
    K_torch = torch.from_numpy(K_np).to(device)
    V_torch = torch.from_numpy(V_np).to(device)

    # for Turing arch, cannot use FlashAttention2
    with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
        O_torch = F.scaled_dot_product_attention(Q_torch, K_torch, V_torch,
                                                 attn_mask=None,  # no masking
                                                 dropout_p=0.0,  # no dropout
                                                 is_causal=causal)

    ##### Use torch profiler to see which CUDA kernel it is using for attention
    # with torch.profiler.profile(
    #     activities=[torch.profiler.ProfilerActivity.CUDA],
    #     record_shapes=True,
    #     with_stack=False
    # ) as prof:
    #     O_torch = F.scaled_dot_product_attention(Q_torch, K_torch, V_torch, attn_mask=None, dropout_p=0.0, is_causal=causal)
    #     torch.cuda.synchronize()
    
    # print(prof.key_averages().table(sort_by="cuda_time_total", max_name_column_width=200))
    # torch.cuda.empty_cache()
    # torch.cuda.reset_peak_memory_stats()
    # torch.cuda.synchronize()
    #####

    return O_torch.cpu().numpy()

In [ ]:
seq_q = seq_kv = 256
d = 64
seed = 42
causal = False

### Create numpy arrays

Must be `batch_size x num_heads x seq_len x head_dim` for memory-efficient attention.

If it is 2D (`seq_len x head_dim` only), torch will fall back to the Math implementation that has no fused operations.

In [ ]:
# Use FP16 for FA1 or MemEff Attention
# Ensure 4D with correct axes to match MemEff implementation
Q_np = generate_matrix((seq_q, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
K_np = generate_matrix((seq_kv, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
V_np = generate_matrix((seq_kv, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
Q_np.shape, K_np.shape, V_np.shape

RTX 2060 and T4 (Turing) use CUDA Kernel:

```
fmha_cutlassF_f16_aligned_64x64_rf_sm75(PyTorchMemEffAttention::AttentionKernel<cutlass::half_t, cutlass::arch::Sm75, true, 64, 64, 64, true, true>::Params)
```

### Run the attention algorithm

In [ ]:
scaled_dot_product_attention(Q_np, K_np, V_np, causal)

FlashAttention currently supports:

Turing, Ampere, Ada, or Hopper GPUs (e.g., H100, A100, RTX 3090, T4, RTX 2080).
fp16 and bf16 (bf16 requires Ampere, Ada, or Hopper GPUs).
Head dimensions that are multiples of 8, up to 128 (e.g., 8, 16, 24, ..., 128). Head dim > 64 backward requires A100 or H100.

## 3. Run Testing Script with NSight

### Test the testing script before profiling

In [ ]:
!.venv/bin/python testing_script.py

### Find the correct kernel and Number of Kernels to skip

It should be the same as the number of warmup kernels in the `testing_script` (e.g. 5)

In [ ]:
!ncu --print-summary per-kernel .venv/bin/python testing_script.py

### Profile the Kernel

#### Gets a lot of metrics that are irrelevant, just make sure it's hitting the correct kernel

In [ ]:
!ncu --set full --launch-skip 5 --launch-count 1 -f -o profile_test_full .venv/bin/python testing_script.py

#### Get the correct metrics to profile

All metrics can be found in the [Metrics Reference documentation](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-reference)

In [ ]:
s = """
dram__cycles_active.avg,
gpu__time_duration.sum,
1tex__cycles_active.avg,
lts__cycles_active.avg,
sm__cycles_active.avg,
sm__cycles_elapsed.avg,
sm__inst_executed.sum,
sm__inst_executed_pipe_lsu.avg,
sm__pipe_alu_cycles_active.avg,
sm__pipe_aluheavy_cycles_active.avg,
sm__pipe_alulite_cycles_active.avg,
sm__pipe_fma_cycles_active.avg,
sm__pipe_fmaheavy_cycles_active.avg,
sm__pipe_fmalite_cycles_active.avg,
sm__pipe_fp16_cycles_active.avg,
sm__pipe_fp64_cycles_active.avg,
sm__pipe_lsu_cycles_active.avg,
sm__pipe_shared_cycles_active.avg,
sm__pipe_tensor_cycles_active.avg,
sm__sass_thread_inst_executed_op_fadd_pred_on.sum,
sm__sass_thread_inst_executed_op_fmul_pred_on.sum,
sm__sass_thread_inst_executed_op_ffma_pred_on.sum,
sm__sass_thread_inst_executed_op_hadd_pred_on.sum,
smsp__inst_executed_pipe_alu.sum,
smsp__inst_executed_pipe_fma.sum,
smsp__inst_executed_pipe_lsu.sum,
smsp__inst_executed_pipe_tensor.sum,
smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed,
smsp__inst_executed_pipe_xu.sum,
smsp__issue_active.avg.pct_of_peak_sustained_elapsed,
smsp__pcsamp_warps_issue_stalled_lg_throttle,
smsp__pcsamp_warps_issue_stalled_long_scoreboard,
smsp__pcsamp_warps_issue_stalled_math_pipe_throttle,
smsp__pcsamp_warps_issue_stalled_not_selected,
smsp__pcsamp_warps_issue_stalled_short_scoreboard,
smsp__warps_issue_stalled_long_scoreboard.avg,
smsp__warps_issue_stalled_math_pipe_throttle.avg,
smsp__warps_issue_stalled_mio_throttle.avg,
smsp__warps_issue_stalled_not_selected.avg,
smsp__warps_issue_stalled_short_scoreboard.avg,
smsp__warps_issue_stalled_wait.avg
"""

metrics_to_measure = s.replace("\n", "").split(",")
metrics_to_measure = [m.strip() for m in metrics_to_measure]
metrics_str = ','.join(metrics_to_measure)
print(metrics_str)


#### Construct commands with correct metrics and parameters

In [ ]:
import torch

_raw = torch.cuda.get_device_name(0).lower()
for _prefix in ["nvidia geforce ", "nvidia ", "geforce ", "tesla ", "quadro "]:
    if _raw.startswith(_prefix):
        _raw = _raw[len(_prefix):]
        break
gpu_name = _raw.replace(" ", "")
print(f"GPU: {torch.cuda.get_device_name(0)} -> slug: {gpu_name}")

In [ ]:
seq_lens = [256, 512, 1024, 2048]
head_dims = [64]
num_runs = 5

In [ ]:
cmds = []

for seq_len in seq_lens:
    for head_dim in head_dims:
        config_dir = f"profiles/{gpu_name}/{seq_len}x{head_dim}"
        os.makedirs(config_dir, exist_ok=True)
        for run_idx in range(num_runs):
            output_name = f"{config_dir}/run{run_idx}"
            cmd = (
                f"ncu --metrics {metrics_str}"
                f" --launch-skip 5 --launch-count 1"
                f" -f -o {output_name}"
                f" .venv/bin/python testing_script.py"
                f" --seq_q {seq_len} --seq_kv {seq_len} --d {head_dim}"
            )
            cmds.append(cmd)
            print(f'{gpu_name} {seq_len}x{head_dim} run {run_idx}:')
            print(cmd + '\n')

#### Run the profiler with the correct metrics

In [ ]:
for cmd in cmds:
    !{cmd}

### View Profile Results

Instructions and example code for profiler API found in the [Python Report Interface documentation](https://docs.nvidia.com/nsight-compute/PythonReportInterface/index.html)

#### Setup package

In [ ]:
!python -m pip install jupyterlab-nvidia-nsight

Add the directory for the `ncu_report` package in NSight to the PYTHON PATH

In [ ]:
import subprocess
result = subprocess.run(['find', '/usr', '/opt', '-name', 'ncu_report*',
'-type', 'f'],
                        capture_output=True, text=True)

path = result.stdout.splitlines()[0][:-len('ncu_report.py')]
# print(path)

import sys                                                                  
sys.path.append(path)
import ncu_report

#### Get the report info

In [ ]:
import pandas as pd

def get_report_metrics(
    report_name: str,
    metrics_names: list | None = None,
    gpu_name: str | None = None,
    seq_len: int | None = None,
    head_dim: int | None = None,
    run_idx: int | None = None,
) -> pd.DataFrame:
    """
    Returns a wide-format DataFrame with one row per (range, action).

    Columns:
        GpuName, SeqLen, HeadDim, Run, Range, Action  -- metadata
        <metric_name>, ...                             -- one column per metric

    If `metrics_names` is provided, only those metrics appear as columns.
    Otherwise all metrics in the report are included.
    """
    _context = ncu_report.load_report(report_name)
    rows = []
    for r in range(_context.num_ranges()):
        _range = _context.range_by_idx(r)
        for a in range(_range.num_actions()):
            _action = _range.action_by_idx(a)
            row = {
                'GpuName': gpu_name,
                'SeqLen': seq_len,
                'HeadDim': head_dim,
                'Run': run_idx,
                'Range': r,
                'Action': _action.name(),
            }
            _names = [name for name in _action.metric_names()
                      if (metrics_names is None or name in metrics_names)]
            for name in _names:
                metric = _action.metric_by_name(name)
                metric_str = metric.as_string()
                metric_float = metric.as_double()
                row[name] = metric_str if (metric_str is not None and metric_float != 0.0) else metric_float
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
dfs = []
for seq_len in seq_lens:
    for head_dim in head_dims:
        for run_idx in range(num_runs):
            df = get_report_metrics(
                f"profiles/{gpu_name}/{seq_len}x{head_dim}/run{run_idx}.ncu-rep",
                metrics_to_measure,
                gpu_name=gpu_name, seq_len=seq_len,
                head_dim=head_dim, run_idx=run_idx,
            )
            dfs.append(df)

all_metrics = pd.concat(dfs, ignore_index=True)
all_metrics.to_csv(f'metrics_{gpu_name}.csv', index=False)

##### Example report content

In [ ]:
REPORT_NAME = r"profiles/rtx2060/256x64/run0.ncu-rep"

my_context = ncu_report.load_report(REPORT_NAME)
my_context.num_ranges()

In [ ]:
my_range = my_context.range_by_idx(0)
my_range.num_actions()

In [ ]:
my_action = my_range.action_by_idx(0)
my_action.name()

In [ ]:
df = get_report_metrics(REPORT_NAME, metrics_to_measure)

# Filter to action of interest
row = df[df['Action'] == my_action.name()].iloc[0]

present = [c for c in metrics_to_measure if c in df.columns]
missing = [m for m in metrics_to_measure if m not in df.columns]

print(f"Missing metrics ({len(missing)}):")
for key in sorted(missing):
    print(f"  {key}")

print(f"\nAvailable metrics ({len(present)}):")
for key in sorted(present):
    print(f"  {key}: {row[key]}")